# 🧠 AI Logic Engine: Knowledge Ingestion (Extreme Debug & Speed Pass)

### 🚀 Update Log (v4.3.1):
- **Firestore Fix:** `database` parameter error ကို `google.cloud.firestore.Client` တိုက်ရိုက်သုံးပြီး ဖြေရှင်းထားပါသည်။
- **Target:** General Knowledge, Greetings, Manners, Social Phrases.
- **Real-time Debug:** AI ဘာတွေဖြေနေလဲ၊ Database မှာ ဘာတွေသိမ်းနေလဲဆိုတာကို Step-by-step ထုတ်ပြပေးပါမယ်。

In [ ]:
# @title 🛠️ Step 1: Install & Imports
import time
print("⏳ Installing libraries...")
!pip install -q firebase-admin google-cloud-firestore transformers accelerate bitsandbytes torch tqdm

import os, json, re, torch, datetime
from tqdm.auto import tqdm
import firebase_admin
from firebase_admin import credentials, firestore
from google.colab import files
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

print("✅ Setup Complete.")

In [ ]:
# @title 🔑 Step 2: Firestore Connection
DATABASE_ID = 'ai-studio-09d97652-b1c3-4b1a-b63e-31d8a38be4c2'

filename = None
print("Service Account Key (JSON) ကို Upload တင်ပေးပါ...")
uploaded = files.upload()
if not uploaded:
    raise ValueError("❌ JSON key လိုအပ်ပါသည်။")
filename = list(uploaded.keys())[0]
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = filename

if not firebase_admin._apps:
    cred = credentials.Certificate(filename)
    firebase_admin.initialize_app(cred)

# 🔥 Fix: Using google.cloud.firestore directly to ensure database support
from google.cloud import firestore as g_firestore
from google.oauth2 import service_account
gcp_credentials = service_account.Credentials.from_service_account_file(filename)
project_id = firebase_admin.get_app().project_id
db = g_firestore.Client(project=project_id, database=DATABASE_ID, credentials=gcp_credentials)

print(f"✅ Firestore Synced (Robust Mode): {DATABASE_ID}")

In [ ]:
# @title 🧬 Step 3: Logic Engine Config
STATE_FILE = "ingestion_state_v4_3.json"

import hashlib
def clean_id(text):
    if not text: return ""
    text = str(text).lower().strip()
    # Support Unicode but replace forbidden characters for Firestore IDs
    if re.search(r'[^\x00-\x7F]', text):
        return 'my_' + hashlib.md5(text.encode('utf-8')).hexdigest()[:16]
    text = re.sub(r'[\s/\\.#\[\]\*\?\!]+', '_', text)
    text = re.sub(r'_{2,}', '_', text)
    return text.strip('_')[:128]

def normalize_verb(v):
    v = str(v).strip().lower()
    is_a_set = ['is_a', 'is a', 'is type of', 'belongs to', 'category of', 'member of']
    if any(p == v for p in is_a_set): return 'is_a'
    return v.replace(' ', '_')

def save_state(count, loop_count=0):
    with open(STATE_FILE, "w") as f: json.dump({"count": count, "loop_count": loop_count, "timestamp": str(datetime.datetime.now())}, f)

def load_state():
    if os.path.exists(STATE_FILE):
        try:
            with open(STATE_FILE, "r") as f:
                d = json.load(f)
                return d.get("count", 0), d.get("loop_count", 0)
        except: pass
    return 0, 0

print("✅ Logic Configured.")

In [ ]:
# @title 🤖 Step 4: Load Faster Model
model_id = "Qwen/Qwen2.5-1.5B-Instruct"
print(f"⏳ Loading {model_id} (Quantized 4-bit)... ")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_compute_dtype=torch.float16, 
    bnb_4bit_quant_type="nf4"
)

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None: 
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    quantization_config=bnb_config, 
    device_map="auto"
)
model.eval()
print(f"✅ Model Loaded on: {model.device}")

In [ ]:
# @title 🚀 Step 5: Start Continuous Ingestion
TARGET_FACT_COUNT = 1000000 # @param {type:"number"}
CATEGORIES = [
    "Myanmar History 1500-1900", "Buddhism core principles", "Myanmar Geography and Rivers",
    "Bagan Dynasty facts", "Modern Myanmar History", "Theravada Buddhism",
    "Famous Myanmar Kings", "Myanmar Festivals", "General Science", "World Geography",
    "Myanmar Traditional Medicine", "Agriculture in Myanmar", "Famous Myanmar Authors",
    "Myanmar Independence", "Buddhist Meditation", "Myanmar Architecture",
    "Space exploration history", "Human Biology basics", "Global Climate",
    "Myanmar Folklore and Myths", "Ethics and Morality", "Myanmar Economics"
]

current_count, loop_count = load_state()
pbar = tqdm(initial=current_count, total=TARGET_FACT_COUNT, desc="Syncing Symbols")

print(f"\n🔥 INGESTION STARTING... (Target Goal: {TARGET_FACT_COUNT})")
print("Logs will show every AI generation to ensure proof of work.\n")

import random
import gc
MAX_BATCH = 400
MIN_TOKEN_LEN = 3
STOP_WORDS = {'the', 'a', 'an', 'it', 'its', 'they', 'them', 'this', 'that'}

def is_valid_triplet(s, v, o):
    if not s or not v or not o: return False
    if len(s) < MIN_TOKEN_LEN or len(o) < MIN_TOKEN_LEN: return False
    if s.lower() in STOP_WORDS or o.lower() in STOP_WORDS: return False
    if s.isdigit() or o.isdigit(): return False
    if s.lower() == o.lower(): return False
    return True

retry_count = 0
MAX_RETRIES = 5

while current_count < TARGET_FACT_COUNT:
    loop_start = time.time()
    try:
        cat = CATEGORIES[loop_count % len(CATEGORIES)]
        
        # Enhanced Diverse Prompt
        prompt = f"<|im_start|>system\nYou are a professional historian and scholar. Extract exactly 20 distinct, deep factual triplets about {cat}. Use format: Subject|Verb|Object|FullSentence. No chat, no numbering, no preamble. Ensure high variety.<|im_end|>\n<|im_start|>user\nGive me 20 diverse factual triplets about {cat}.<|im_end|>\n<|im_start|>assistant\n"
        
        # 1. Tokenizing
        inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)
        
        print(f"[{datetime.datetime.now().strftime('%H:%M:%S')}] ⏳ AI generating facts for: {cat}...", end=" ")
        
        # 2. Generating
        with torch.no_grad():
            out = model.generate(
                inputs.input_ids, 
                attention_mask=inputs.attention_mask, 
                max_new_tokens=512, 
                do_sample=True, 
                temperature=0.7,
                pad_token_id=tokenizer.pad_token_id,
                use_cache=True
            )
        
        # 3. Decoding
        response = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()
        del inputs, out
        
        if not response: 
            print("❌ AI returned empty response. Retrying...")
            continue
            
        print("✅ Received.")
        retry_count = 0
        
        # 4. Parsing & DB Write
        batch = db.batch()
        added_this_loop = 0
        batch_count = 0
        lines = response.split('\n')
        
        for line in lines:
            line = line.strip()
            # clean off list numbers or dashes
            line = re.sub(r"^[0-9]+\.*\s*", "", line)
            line = re.sub(r"^[\-\+\*]\s*", "", line)
            if '|' not in line: continue
            parts = [p.strip() for p in line.split('|')]
            
            if len(parts) >= 3:
                s, v, o = parts[0], normalize_verb(parts[1]), parts[2]
                sid, oid = clean_id(s), clean_id(o)
                
                if sid and oid and sid != oid and is_valid_triplet(s, v, o):
                    node_ref = db.collection('nodes').document(sid)
                    data = {'id': sid, 'name': s, 'type': 'ENTITY', 'updatedAt': g_firestore.SERVER_TIMESTAMP}
                    
                    if v == 'is_a':
                        data['groups'] = g_firestore.ArrayUnion([oid])
                    else:
                        sent = parts[3] if len(parts) > 3 else f"{s} {v.replace('_',' ')} {o}."
                        data['relations'] = g_firestore.ArrayUnion([{
                            'verb': v, 'targetId': oid, 'sentence': sent, 'weight': 50
                        }])
                    
                    batch.set(node_ref, data, merge=True)
                    added_this_loop += 1
                    batch_count += 1
                    
                    if batch_count >= MAX_BATCH:
                        batch.commit()
                        batch = db.batch()
                        batch_count = 0
        
        # 5. Commit Check
        if batch_count > 0:
            print(f"   ☁️ Syncing remaining {batch_count} symbols to Firestore...", end=" ")
            batch.commit()
            print("Done.")
        
        if added_this_loop > 0:
            loop_count += 1
            current_count += added_this_loop
            pbar.update(added_this_loop)
            save_state(current_count, loop_count)
        else:
            print(f"   ⚠️ No valid triplets found for {cat}. Response was: \n{response[:200]}...")
            
        # 6. Memory Maintenance
        if current_count % 30 == 0: 
            torch.cuda.empty_cache()
            gc.collect()
            
    except KeyboardInterrupt:
        print("\n🛑 Manually Stopped. Finalizing state...")
        break
    except Exception as e:
        print(f"\n❌ [SYSTEM ERROR] {str(e)}")
        wait = (2 ** retry_count) + random.uniform(0, 1)
        print(f"⏳ Retry {retry_count+1}/{MAX_RETRIES} after {wait:.1f}s...")
        time.sleep(wait)
        retry_count += 1
        if retry_count >= MAX_RETRIES: break

pbar.close()
print(f"✅ Finished. Total Concepts in Database: {current_count}")